In [ ]:
# Import external settings and database connection functions from another notebook
%run "00_globals_and_db.ipynb"

# Establish a connection to the database using a custom helper function
cnx = db_connect()
# Create a cursor object, which allows us to execute SQL commands
cur = cnx.cursor()
# Notify user that the database connection is active
print("Connected.")

# Define a list of table names and SQL queries to get their current row counts
queries = [
    ("raw_case_row", "SELECT COUNT(*) FROM raw_case_row"),
    ("raw_pdf", "SELECT COUNT(*) FROM raw_pdf"),
    ("raw_pdf_text", "SELECT COUNT(*) FROM raw_pdf_text"),
    ("case_seller_or_service_provider", "SELECT COUNT(*) FROM case_seller_or_service_provider"),
    ("core_case_resolution_event", "SELECT COUNT(*) FROM core_case_resolution_event"),
]

# Loop through each query in the list, execute it, and print the results
for name, q in queries:
    # Execute the specific COUNT query
    cur.execute(q)
    # fetchone()[0] gets the first column (the count) from the first result row
    print(f"{name}: {cur.fetchone()[0]}")

# Identify PDF URLs in 'raw_case_row' that do NOT exist in 'raw_pdf' yet
cur.execute('''
    SELECT r.row_id, r.pdf_url_hash, r.pdf_url
    FROM raw_case_row r
    LEFT JOIN raw_pdf p ON p.pdf_url_hash = r.pdf_url_hash
    WHERE p.pdf_id IS NULL
    ORDER BY r.row_id ASC
''')
# Fetch all records that still need to be downloaded
pending_download = cur.fetchall()
# Print the total count of files to be downloaded
print("Pending downloads:", len(pending_download))
# Preview the first 5 records for debugging
print("Sample (up to 5):", pending_download[:5])

# Define the file path for the output JSONL file using a path variable
pending_download_path = DIR_RAW / "pending_download.jsonl"
# Open the file for writing with UTF-8 encoding
with pending_download_path.open("w", encoding="utf-8") as f:
    # Iterate through each record found in the previous query
    for row_id, pdf_url_hash, pdf_url in pending_download:
        # Convert row data to JSON and write as a single line in the file
        f.write(json.dumps({
            "row_id": int(row_id),
            "pdf_url_hash": pdf_url_hash,
            "pdf_url": pdf_url,
        }, ensure_ascii=False) + "\n")
# Print the location where the download list was saved
print("Wrote:", pending_download_path)

# Find downloaded PDFs that are missing from at least one processed data table
cur.execute('''
    SELECT p.pdf_id, p.row_id, p.pdf_url, p.pdf_url_hash, p.sha256, p.local_path
    FROM raw_pdf p
    LEFT JOIN raw_pdf_text t ON t.sha256 = p.sha256
    LEFT JOIN case_seller_or_service_provider s ON s.seller_or_service_provider_ID = COALESCE(p.row_id, p.pdf_id)
    LEFT JOIN case_consumer_person c ON c.consumer_person_ID = COALESCE(p.row_id, p.pdf_id)
    LEFT JOIN case_dispute d ON d.case_dispute_ID = COALESCE(p.row_id, p.pdf_id)
    LEFT JOIN case_resolution r ON r.case_resolution_ID = COALESCE(p.row_id, p.pdf_id)
    LEFT JOIN core_case_resolution_event e ON e.case_resolution_event_ID = COALESCE(p.row_id, p.pdf_id)
    WHERE t.sha256 IS NULL
       OR s.seller_or_service_provider_ID IS NULL
       OR c.consumer_person_ID IS NULL
       OR d.case_dispute_ID IS NULL
       OR r.case_resolution_ID IS NULL
       OR e.case_resolution_event_ID IS NULL
    ORDER BY p.downloaded_at ASC
''')
# Store all rows that require parsing/processing
pending_parse = cur.fetchall()
# Print total number of files requiring parsing
print("Pending parses:", len(pending_parse))
# Print preview of the first 5 entries (slicing each row to show 5 fields only)
print("Sample (up to 5):", [x[:5] for x in pending_parse[:5]])

# Define the output path for the parsing tasks
pending_parse_path = DIR_OUT / "pending_parse.jsonl"
# Open and write the pending parsing data to a JSONL file
with pending_parse_path.open("w", encoding="utf-8") as f:
    # Iterate through the database results
    for pdf_id, row_id, pdf_url, pdf_url_hash, sha256, local_path in pending_parse:
        # Convert record to JSON format and write as a line
        f.write(json.dumps({
            "pdf_id": int(pdf_id),
            "row_id": int(row_id) if row_id is not None else None,
            "pdf_url": pdf_url,
            "pdf_url_hash": pdf_url_hash,
            "sha256": sha256,
            "local_path": local_path,
        }, ensure_ascii=False) + "\n")
# Print the final path of the generated parse list
print("Wrote:", pending_parse_path)

# Count how many groups of duplicate URL hashes exist in the source table
cur.execute('''
    SELECT COUNT(*) 
    FROM (
      SELECT pdf_url_hash, COUNT(*) c
      FROM raw_case_row
      GROUP BY pdf_url_hash
      HAVING c > 1
    ) x
''')
# Print the duplicate count for raw_case_row
print("raw_case_row duplicate pdf_url_hash groups:", cur.fetchone()[0])

# Count how many groups of duplicate URL hashes exist in the PDF storage table
cur.execute('''
    SELECT COUNT(*)
    FROM (
      SELECT pdf_url_hash, COUNT(*) c
      FROM raw_pdf
      GROUP BY pdf_url_hash
      HAVING c > 1
    ) x
''')
# Print the duplicate count for raw_pdf
print("raw_pdf duplicate pdf_url_hash groups:", cur.fetchone()[0])

# Close the database cursor to prevent memory leaks
cur.close()
# Terminate the database connection
cnx.close()
# Final completion message
print("Done.")